# Medical Image Segmentation on Kaggle (GPU)
U-Net vs Attention U-Net vs U-Net++ on ISIC-2018 skin lesions.

**Before running:** Settings → Accelerator → **GPU**; Add Input → the ISIC-2018
segmentation dataset; and add your code (Method A: your uploaded code dataset,
or Method B: GitHub). See `KAGGLE_GUIDE.md`.

## Cell 1 — GPU check + install dependencies

In [ ]:
!nvidia-smi
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8 grad-cam medpy timm

## Cell 2 — Inspect the ISIC dataset folder tree

In [ ]:
import os
INPUT = '/kaggle/input'
for root, dirs, files in os.walk(INPUT):
    depth = root.replace(INPUT, '').count(os.sep)
    if depth > 3:
        continue
    n_jpg = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg'))])
    n_png = len([f for f in files if f.lower().endswith('.png')])
    tag = f'  [jpg={n_jpg} png={n_png}]' if (n_jpg or n_png) else ''
    print('  ' * depth + os.path.basename(root) + '/' + tag)

## Cell 3A — Bring code in (Method A: uploaded code dataset)
Set `CODE_DATASET` to your dataset folder name under /kaggle/input.

In [ ]:
import shutil, os, glob
CODE_DATASET = '/kaggle/input/medical-seg-code'   # <-- change to your dataset path
WORK = '/kaggle/working/medseg'
os.makedirs(WORK, exist_ok=True)
# If your zip nested the project one level deep, point src_root at that folder.
src_root = CODE_DATASET
if not os.path.exists(os.path.join(src_root, 'src')):
    cand = glob.glob(os.path.join(CODE_DATASET, '*', 'src'))
    if cand:
        src_root = os.path.dirname(cand[0])
for item in ['src', 'app', 'configs', 'tests']:
    s = os.path.join(src_root, item)
    if os.path.exists(s):
        shutil.copytree(s, os.path.join(WORK, item), dirs_exist_ok=True)
for d in ['data/splits', 'checkpoints', 'outputs']:
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
print('code copied to', WORK)
print(os.listdir(WORK))

## Cell 3B — Bring code in (Method B: GitHub) — use *instead of* 3A

In [ ]:
# Requires Settings -> Internet -> On
# !git clone https://github.com/<you>/medical-image-segmentation.git /kaggle/working/medseg
# WORK = '/kaggle/working/medseg'

## Cell 4 — Auto-detect ISIC image/mask folders, patch config, build splits

In [ ]:
import os, sys, yaml, glob
WORK = '/kaggle/working/medseg'
os.chdir(WORK)
sys.path.insert(0, WORK)

# Find the folder with the most .jpg (images) and the most *segmentation*.png (masks)
def best_dir(input_root, predicate):
    best, best_n = None, 0
    for root, _, files in os.walk(input_root):
        n = sum(1 for f in files if predicate(f))
        if n > best_n:
            best, best_n = root, n
    return best, best_n

img_dir, n_img = best_dir('/kaggle/input', lambda f: f.lower().endswith(('.jpg', '.jpeg')))
mask_dir, n_mask = best_dir('/kaggle/input', lambda f: f.lower().endswith('.png'))
print(f'images: {img_dir}  ({n_img})')
print(f'masks : {mask_dir}  ({n_mask})')

# Patch config.yaml to point at the detected Kaggle paths.
cfg_path = os.path.join(WORK, 'configs', 'config.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg['project']['device'] = 'cuda'
cfg['data']['images_dir'] = img_dir
cfg['data']['masks_dir'] = mask_dir
# ISIC masks are named <id>_segmentation.png; if your dataset differs, edit mask_suffix.
with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f)
print('config patched.')

# Build train/val/test splits
from src.dataset import make_splits
make_splits(img_dir, mask_dir, os.path.join(WORK, 'data', 'splits'),
            mask_suffix=cfg['data']['mask_suffix'],
            val_fraction=cfg['data']['val_fraction'],
            test_fraction=cfg['data']['test_fraction'],
            seed=cfg['project']['seed'])

## Cell 5 — Quick sanity run (3 epochs) then full U-Net baseline

In [ ]:
# Smoke test (3 epochs) — confirm Dice climbs before committing to 60.
!python -m src.train --arch unet --epochs 3

In [ ]:
# Full baseline (uses epochs from config.yaml, default 60, early-stops)
!python -m src.train --arch unet

## Cell 6 — Train the two variants

In [ ]:
!python -m src.train --arch attention_unet
!python -m src.train --arch unetpp

## Cell 7 — Evaluate + comparison table (for the report)

In [ ]:
!python -m src.evaluate --compare \
  checkpoints/best_unet.pt \
  checkpoints/best_attention_unet.pt \
  checkpoints/best_unetpp.pt
import pandas as pd
pd.read_csv('outputs/comparison.csv')

## Cell 8 — Visualise predictions

In [ ]:
import cv2, pandas as pd, matplotlib.pyplot as plt
from src.predict import SegmentationPredictor
from src.utils import resolve

predictor = SegmentationPredictor('checkpoints/best_attention_unet.pt', 'configs/config.yaml')
df = pd.read_csv('data/splits/test.csv')
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax_row, (_, row) in zip(axes, df.sample(3, random_state=1).iterrows()):
    img = cv2.cvtColor(cv2.imread(str(resolve(row['image_path']))), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(str(resolve(row['mask_path'])), cv2.IMREAD_GRAYSCALE)
    overlay, mask, area = predictor.predict_overlay(img)
    ax_row[0].imshow(img); ax_row[0].set_title('image'); ax_row[0].axis('off')
    ax_row[1].imshow(gt, cmap='gray'); ax_row[1].set_title('ground truth'); ax_row[1].axis('off')
    ax_row[2].imshow(overlay); ax_row[2].set_title(f'pred ({area*100:.1f}%)'); ax_row[2].axis('off')
plt.tight_layout(); plt.show()

## Cell 9 — Download checkpoints
Files in `/kaggle/working/medseg/checkpoints/` appear under the notebook's
**Output** tab (right panel) for download. Or **Save Version** to persist them.

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/export', exist_ok=True)
for f in os.listdir('checkpoints'):
    shutil.copy(os.path.join('checkpoints', f), '/kaggle/working/export/')
for f in ['outputs/comparison.csv']:
    if os.path.exists(f):
        shutil.copy(f, '/kaggle/working/export/')
print('ready to download:', os.listdir('/kaggle/working/export'))